# Evals with LangChain: a visual, offline experiment

*Companion to `evals_learning.ipynb`. Same ideas, shown in LangChain / LangSmith / openevals vocabulary, **with a picture for every step**. Runs offline, no API keys.*

**Slide deck:** https://claude.ai/artifact/Rvbz1Na7RMnTUU8ZBRGmx8

## The whole notebook in one analogy: an exam

| In an exam | In an eval experiment |
|---|---|
| The **exam paper** with an answer key | the **dataset**: questions (`inputs`) plus the right answers (`reference outputs`) |
| The **student** | the **target**: the system you are testing |
| The student's **written answers** | the target's **outputs** |
| The **markers**, each checking one thing (spelling, facts, tone) | the **evaluators**, each producing one score |
| A marked paper with a tick or cross per item | the **feedback table** |
| The **final grade** across the whole class | the **summary evaluators** |
| **Retaking** the exam 3 times to see if results are stable | `num_repetitions=3` |
| Comparing last term's class with this term's | comparing two **experiments** (v1 versus v2) |

Keep this table in mind. Every cell below is one row of it.

## What you will be able to do
1. Draw and explain how an experiment flows from dataset to final score.
2. Run one with `langsmith.evaluate()` and `openevals`, fully offline.
3. Read results as **pictures**: heatmaps, before/after charts, repetition dots, confidence intervals.
4. Spot a biased LLM judge visually.
5. Know what connecting to LangSmith adds, and what to protect first.

## Colour legend used in every chart
**Green** = pass / good. **Red** = fail / bad. **Orange** = needs attention (verbose or suspicious). **Blue** = version v1. **Teal** = version v2.

In [ ]:
# Keep EVERYTHING local. These must be set before importing langsmith / langchain.
import os
os.environ["LANGSMITH_TRACING"] = "false"
os.environ["LANGSMITH_TRACING_SAMPLING_RATE"] = "0"     # drop every trace instead of sending it
os.environ["LANGCHAIN_TRACING_SAMPLING_RATE"] = "0"     # same setting under the older LANGCHAIN_ name
os.environ.pop("LANGSMITH_API_KEY", None)               # no key, so nothing can authenticate to the cloud even by accident

import warnings; warnings.filterwarnings("ignore")      # silence the "beta" notice for upload_results=False
import uuid, datetime, hashlib, re, json
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch, Rectangle, Patch
from collections import defaultdict

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda
from langchain_core.messages import AIMessage
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.output_parsers import StrOutputParser
from langsmith import evaluate
from langsmith.schemas import Example
from openevals.exact import exact_match
from openevals.llm import create_llm_as_judge

from evals import bootstrap_ci, tpr_tnr, cohen_kappa        # our own helpers from the main course

# One colour palette for the whole notebook, so colours always mean the same thing.
C = {"pass": "#3A9D6E", "fail": "#D1495B", "warn": "#E8A33D", "v1": "#4C78A8", "v2": "#1E8E8E",
     "ink": "#14212B", "soft": "#EAF0EE", "grey": "#9AA7AE"}
plt.rcParams.update({"font.size": 11, "axes.spines.top": False, "axes.spines.right": False})

**A note on drawing.** Several figures below are *diagrams*, not data charts. The two small helpers in the next cell draw boxes and arrows. You do not need to understand them; they only make the pictures.

In [ ]:
def box(ax, x, y, w, h, text, color, fs=11, tc=None, bold=True):
    """Draw a rounded box with centred text."""
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.02,rounding_size=0.12", fc=color, ec=C["ink"], lw=1.4))
    ax.text(x + w / 2, y + h / 2, text, ha="center", va="center", fontsize=fs, color=tc or C["ink"],
            fontweight="bold" if bold else "normal", linespacing=1.35)

def arrow(ax, x1, y1, x2, y2, label=None, color=None, rad=0.0, fs=9, dy=0.18):
    """Draw an arrow, optionally curved (rad) and labelled."""
    color = color or C["ink"]
    ax.add_patch(FancyArrowPatch((x1, y1), (x2, y2), arrowstyle="-|>", mutation_scale=16, lw=1.7, color=color,
                                 connectionstyle=f"arc3,rad={rad}"))
    if label:
        ax.text((x1 + x2) / 2, (y1 + y2) / 2 + dy, label, ha="center", fontsize=fs, color=color, style="italic")

def canvas(w=13, h=3.6, xlim=(0, 14), ylim=(0, 4)):
    fig, ax = plt.subplots(figsize=(w, h)); ax.set_xlim(*xlim); ax.set_ylim(*ylim); ax.axis("off")
    return fig, ax

## 0. The big picture: how one experiment flows

**Read this diagram left to right.** Every experiment, in any tool, has this shape.

- The **dataset** feeds **inputs** into the **target**.
- The target produces **outputs**.
- Each **evaluator** compares the outputs with the dataset's **reference outputs** (the arc over the top) and writes one **score**.
- All scores land in one **feedback table**.
- **Summary evaluators** look at the whole table and produce overall numbers.

In [ ]:
fig, ax = canvas(13.5, 4.6, (0, 14), (0, 4.8))
names = [("DATASET", "12 examples\ninputs + answers", C["soft"], "exam paper\n+ answer key"),
         ("TARGET", "your system\n(chain / agent)", "#DCE8F5", "the student"),
         ("OUTPUTS", "answer, intent", "#DCE8F5", "written answers"),
         ("EVALUATORS", "3 functions,\none score each", "#FBE9C8", "the markers"),
         ("FEEDBACK", "table of scores\n(per example)", "#D8EFE3", "marked paper"),
         ("SUMMARY", "overall\nnumbers", "#D8EFE3", "final grade")]
xs = [0.2 + i * 2.3 for i in range(6)]
for (title, sub, col, analogy), x in zip(names, xs):
    box(ax, x, 1.3, 1.9, 1.5, f"{title}\n{sub}", col, fs=9)
    ax.text(x + 0.95, 0.95, f"({analogy})", ha="center", va="top", fontsize=8.5, color=C["grey"], style="italic")
for i in range(5):
    arrow(ax, xs[i] + 1.9, 2.05, xs[i + 1], 2.05)
arrow(ax, xs[0] + 0.95, 2.8, xs[3] + 0.95, 2.8, color=C["warn"], rad=-0.35)
ax.text(xs[1] + 1.9, 4.25, "reference outputs travel here too (the answer key)", ha="center", fontsize=9.5, color="#B87A14", style="italic")
ax.text(xs[1] + 0.95, 0.25, "num_repetitions=3: the target runs each example 3 times", ha="center", fontsize=9, color=C["v2"], fontweight="bold")
ax.set_title("One experiment, end to end", fontsize=13, fontweight="bold", loc="left")
plt.show()

## 1. The system under test: a LangChain "support bot"

A **target** is anything that maps `inputs -> outputs`. Here it is a small LangChain **chain**: `prompt | model | parser`.
The `|` symbol pipes the output of one step into the next, like an assembly line. The picture:

In [ ]:
fig, ax = canvas(13.5, 3.4, (0, 14), (0, 3.4))
steps = [("PROMPT\ntemplate", C["soft"]), ("MODEL\n(stub here)", "#DCE8F5"), ("PARSER", C["soft"])]
sx = [3.0, 6.2, 9.4]
ax.text(0.2, 1.7, '"How long do I\nhave to return\nan item?"', va="center", fontsize=9.5, family="monospace", bbox=dict(fc="white", ec=C["grey"], boxstyle="round"))
for (t, col), x in zip(steps, sx):
    box(ax, x, 1.1, 2.0, 1.2, t, col, fs=11)
ax.text(12.0, 1.7, '"You can return\nitems within\n60 days."', va="center", fontsize=9.5, family="monospace", bbox=dict(fc="#FBE3E6", ec=C["fail"], boxstyle="round"))
arrow(ax, 2.0, 1.7, 3.0, 1.7); arrow(ax, 5.0, 1.7, 6.2, 1.7); arrow(ax, 8.2, 1.7, 9.4, 1.7); arrow(ax, 11.4, 1.7, 11.95, 1.7)
ax.text(4.0, 0.7, "wraps the question\nin a chat message", ha="center", fontsize=8.5, color=C["grey"], style="italic")
ax.text(7.2, 0.7, "real model goes here:\nChatAnthropic / ChatOpenAI", ha="center", fontsize=8.5, color=C["grey"], style="italic")
ax.text(10.4, 0.7, "turns the reply into\nplain text", ha="center", fontsize=8.5, color=C["grey"], style="italic")
ax.set_title("The chain: prompt | model | parser", fontsize=13, fontweight="bold", loc="left")
plt.show()

**Why a stub model?** So this notebook is free, offline and gives the same answers every time. The eval does not care what is inside the target, so **swapping in a real model changes nothing else**.

We build two versions so we can compare **before and after**:
- **v1** has planted bugs: it hallucinates two policies (60 days, two years), gives a wrong refund answer, and mislabels one intent.
- **v2** fixes those, but adds a **long non-answer** and one **flaky** answer (right sometimes, wrong other times).

In [ ]:
# ---- The dataset: question, reference phrase the answer must contain, and the expected intent. ----
DATASET = [
    {"question": "How long do I have to return an item?",       "ref": "30 days",                 "intent": "returns"},
    {"question": "How long does standard shipping take?",       "ref": "3 to 5 business days",    "intent": "shipping"},
    {"question": "How will my refund be paid?",                 "ref": "original payment method", "intent": "refunds"},
    {"question": "How long is the warranty?",                   "ref": "one year",                "intent": "warranty"},
    {"question": "Do gift cards expire?",                       "ref": "never expire",            "intent": "gift_cards"},
    {"question": "How long does international delivery take?",  "ref": "10 to 14 days",           "intent": "shipping"},
    {"question": "Can I return an opened item?",                "ref": "unused",                  "intent": "returns"},
    {"question": "Will I pay customs fees on overseas orders?", "ref": "customs fees",            "intent": "shipping"},
    {"question": "How do I turn on two-factor authentication?", "ref": "Settings > Security",     "intent": "security"},
    {"question": "I forgot my password, what do I do?",         "ref": "Forgot password",         "intent": "account"},
    {"question": "Can I cancel an order?",                      "ref": "before it ships",         "intent": "orders"},
    {"question": "Can I use a discount code with a gift card?", "ref": "combined",                "intent": "gift_cards"},
]
FLAKY_Q = "How do I turn on two-factor authentication?"
SHORT = ["return window", "standard shipping", "refund method", "warranty length", "gift card expiry", "intl delivery",
         "opened returns", "customs fees", "2FA setup", "password reset", "cancel order", "code + gift card"]   # short names for charts
QSHORT = {d["question"]: s for d, s in zip(DATASET, SHORT)}

# ---- Canned model behavior. Each entry: (answer, intent). v1 has three wrong answers and one wrong intent. ----
V1 = {
    "How long do I have to return an item?":       ("You can return items within 60 days.", "returns"),                       # hallucinated
    "How long does standard shipping take?":       ("Standard shipping takes 3 to 5 business days.", "shipping"),
    "How will my refund be paid?":                 ("Refunds are issued as store credit.", "refunds"),                        # wrong
    "How long is the warranty?":                   ("The warranty lasts two years.", "warranty"),                            # hallucinated
    "Do gift cards expire?":                       ("Gift cards never expire.", "gift_cards"),
    "How long does international delivery take?":  ("International delivery takes 10 to 14 days.", "shipping"),
    "Can I return an opened item?":                ("Returns are accepted for unused items.", "returns"),
    "Will I pay customs fees on overseas orders?": ("Yes, customs fees may apply.", "returns"),                              # wrong intent
    "How do I turn on two-factor authentication?": ("Go to Settings > Security to turn it on.", "security"),
    "I forgot my password, what do I do?":         ("Click Forgot password on the login page.", "account"),
    "Can I cancel an order?":                      ("You can cancel before it ships.", "orders"),
    "Can I use a discount code with a gift card?": ("Yes, they can be combined at checkout.", "gift_cards"),
}
V2 = dict(V1)
V2["How long do I have to return an item?"] = ("You can return items within 30 days of purchase.", "returns")        # fixed
V2["How long is the warranty?"] = ("Our warranty covers manufacturing defects for one year.", "warranty")             # fixed
V2["How will my refund be paid?"] = ("Refund handling depends on a wide variety of factors including the product category, "
                                      "the timing of the request, the region you live in, and several other considerations "
                                      "that our team reviews on a case by case basis.", "refunds")                      # long, but no answer
V2["Will I pay customs fees on overseas orders?"] = ("Yes, customs fees may apply.", "shipping")                        # intent fixed

_calls = defaultdict(int)     # counts calls per question so we can simulate a FLAKY answer deterministically

def _flaky_wrong(question):
    """True on roughly one call in three for the flaky question, decided by a hash so reruns are reproducible."""
    _calls[question] += 1
    return int(hashlib.md5(f"{question}-{_calls[question]}".encode()).hexdigest(), 16) % 3 == 0

def make_bot(table, flaky_question=None):
    """Build the chain prompt | stub model | parser, plus a target function wrapping it."""
    def stub_model(prompt_value):
        question = prompt_value.to_messages()[-1].content
        answer, _ = table[question]
        if question == flaky_question and _flaky_wrong(question):
            answer = "Sorry, I am not sure."                        # the flaky failure
        return AIMessage(content=answer)

    chain = ChatPromptTemplate.from_messages([("system", "You are a support assistant."), ("human", "{question}")]) \
            | RunnableLambda(stub_model) | StrOutputParser()

    def target(inputs: dict) -> dict:
        # What LangSmith calls once per example. It receives the example's `inputs` and returns the outputs dict.
        return {"answer": chain.invoke({"question": inputs["question"]}), "intent": table[inputs["question"]][1]}
    return target

target_v1 = make_bot(V1)
target_v2 = make_bot(V2, flaky_question=FLAKY_Q)
print(target_v1({"question": "How long do I have to return an item?"}))

### A map of the planted bugs

Before running anything, here is what we **deliberately** broke. Each row is a question; each column is a version. When the results come in, you can check whether the eval *found* these.

In [ ]:
def category(table, q, ref, intent, flaky=False):
    ans = table[q][0]
    if flaky:                      return ("flaky", C["warn"])
    if ref.lower() in ans.lower():
        return ("wrong intent", "#8E6BBE") if table[q][1] != intent else ("correct", C["pass"])
    if len(ans.split()) > 20:      return ("long non-answer", C["warn"])
    return ("wrong", C["fail"])

fig, ax = plt.subplots(figsize=(8.4, 5.6))
for r, d in enumerate(DATASET):
    for c, (table, flaky) in enumerate([(V1, False), (V2, d["question"] == FLAKY_Q)]):
        label, col = category(table, d["question"], d["ref"], d["intent"], flaky)
        ax.add_patch(Rectangle((c, 11 - r), 0.96, 0.9, fc=col, ec="white", lw=2))
        ax.text(c + 0.48, 11 - r + 0.45, label, ha="center", va="center", color="white", fontsize=9.5, fontweight="bold")
    ax.text(-0.05, 11 - r + 0.45, SHORT[r], ha="right", va="center", fontsize=10.5)
ax.set_xlim(-1.0, 2); ax.set_ylim(0, 13.1); ax.axis("off")
ax.text(0.48, 12.15, "v1", ha="center", fontsize=12, fontweight="bold", color=C["v1"])
ax.text(1.48, 12.15, "v2", ha="center", fontsize=12, fontweight="bold", color=C["v2"])
ax.text(-1.0, 12.75, "Planted behaviours (what the eval should discover)", fontsize=12, fontweight="bold", va="center")
plt.show()

**What to notice.** v1 has 3 red cells (wrong answers) and 1 purple cell (right answer, wrong *intent label*). v2 fixes the two hallucinations and the intent, but its refund answer is now a **long non-answer** (orange) and the two-factor question is **flaky** (orange).
We will see whether the eval finds each of these.

## 2. The dataset: examples with inputs and reference outputs

LangSmith evaluates over **`Example`** objects. One example is one row of the exam paper:

- **`inputs`**: what the target receives.
- **`outputs`**: the **reference** answer (the answer key). Evaluators compare the target's output against this.

In [ ]:
dataset_id = uuid.uuid4()
examples = [
    Example(id=uuid.uuid4(), dataset_id=dataset_id,
            inputs={"question": d["question"]},
            outputs={"answer": d["ref"], "intent": d["intent"]},            # reference outputs, for evaluators to compare against
            created_at=datetime.datetime.now(datetime.timezone.utc))
    for d in DATASET
]
print(len(examples), "examples")

In [ ]:
# One example drawn as a "card", so you can see what each part is for.
fig, ax = canvas(12.5, 3.8, (0, 13), (0, 4))
ex = examples[0]
ax.add_patch(FancyBboxPatch((0.3, 0.4), 7.4, 3.1, boxstyle="round,pad=0.03,rounding_size=0.15", fc="white", ec=C["ink"], lw=1.6))
ax.text(0.6, 3.2, "EXAMPLE #1", fontsize=11, fontweight="bold", va="top")
ax.text(0.6, 2.55, "inputs", fontsize=10, color=C["v1"], fontweight="bold")
ax.text(1.9, 2.55, json.dumps(ex.inputs), fontsize=8.5, family="monospace")
ax.text(0.6, 1.6, "outputs", fontsize=10, color=C["pass"], fontweight="bold")
ax.text(1.9, 1.6, json.dumps(ex.outputs), fontsize=8.5, family="monospace")
ax.text(0.6, 0.9, "(the reference: what a good answer contains)", fontsize=8.5, color=C["grey"], style="italic")
box(ax, 9.2, 2.35, 3.4, 0.9, "TARGET receives\ninputs only", "#DCE8F5", fs=10)
box(ax, 9.2, 0.7, 3.4, 0.9, "EVALUATORS also receive\nthe reference outputs", "#FBE9C8", fs=10)
arrow(ax, 7.75, 2.7, 9.2, 2.8, color=C["v1"]); arrow(ax, 7.75, 1.7, 9.2, 1.2, color=C["pass"])
ax.set_title("What an Example holds, and who sees which part", loc="left", fontsize=12, fontweight="bold")
plt.show()

**Why do the evaluators get the reference but the target does not?** Exactly like an exam: the student must not see the answer key, but the marker needs it.

(In the SDK version used here, a local run needs real `Example` objects. Plain dictionaries failed.)

## 3. Evaluators: small functions that give a score

An **evaluator** is just a function. LangSmith reads its **parameter names** and passes the matching values. This picture shows the contract:

In [ ]:
fig, ax = canvas(13.5, 4.4, (0, 14.5), (0, 4.6))
box(ax, 0.2, 3.0, 3.0, 0.8, "inputs\n(from the dataset)", "#DCE8F5", fs=10)
box(ax, 0.2, 1.7, 3.0, 0.8, "outputs\n(from the target)", "#DCE8F5", fs=10)
box(ax, 0.2, 0.4, 3.0, 0.8, "reference_outputs\n(from the dataset)", "#FBE9C8", fs=10)
box(ax, 5.2, 1.2, 4.0, 1.8, "YOUR EVALUATOR\ndef mentions_reference(\n  outputs, reference_outputs):", C["soft"], fs=10)
box(ax, 10.6, 1.2, 3.7, 1.8, '{"key": "mentions_reference",\n "score": True}', "#D8EFE3", fs=9)
arrow(ax, 3.2, 3.4, 5.2, 2.7); arrow(ax, 3.2, 2.1, 5.2, 2.1); arrow(ax, 3.2, 0.8, 5.2, 1.5)
arrow(ax, 9.2, 2.1, 10.6, 2.1, label="returns")
ax.text(7.2, 4.25, "only the parameters you name are passed in", fontsize=10, color=C["grey"], style="italic", ha="center")
ax.set_title("The evaluator contract", loc="left", fontsize=12, fontweight="bold")
plt.show()

We build four evaluators, each checking **one** thing:

| # | Evaluator | Checks | Type |
|---|---|---|---|
| 1 | `mentions_reference` | does the answer contain the reference phrase? | code |
| 2 | `intent_correct` | is the intent label exactly right? (`openevals` `exact_match`) | code |
| 3 | `judge_correct` | does an **LLM judge** think the answer is correct? (`openevals` `create_llm_as_judge`) | model |
| 4 | `overall_mentions_rate` | the average of evaluator 1 across the whole experiment | summary |

In [ ]:
# ---- 1. A code evaluator ----
def mentions_reference(outputs: dict, reference_outputs: dict) -> dict:
    ok = reference_outputs["answer"].lower() in outputs["answer"].lower()
    return {"key": "mentions_reference", "score": ok}

# ---- 2. openevals exact match, wrapped so it receives plain strings ----
def intent_correct(outputs: dict, reference_outputs: dict):
    return exact_match(outputs=outputs["intent"], reference_outputs=reference_outputs["intent"])   # returns key "exact_match"

# ---- 3. An LLM judge. The model is a stub so we can run offline. ----
class StubJudgeModel(BaseChatModel):
    """Stands in for ChatAnthropic / ChatOpenAI. openevals only needs `.with_structured_output(schema).invoke(messages)`
    to return {"reasoning": ..., "score": ...}, so we implement exactly that, with a lenient, length-biased rule."""
    @property
    def _llm_type(self) -> str:
        return "stub-judge"

    def _generate(self, messages, stop=None, run_manager=None, **kwargs):      # required by the base class, unused here
        raise NotImplementedError

    def with_structured_output(self, schema, **kwargs):
        def judge(messages):
            text = " ".join(m["content"] if isinstance(m, dict) else m.content for m in messages)
            answer = re.search(r"Answer: (.*)", text).group(1)
            reference = re.search(r"Reference: (.*)", text).group(1)
            correct = reference.lower() in answer.lower()
            long_and_confident = len(answer.split()) > 20                      # the verbosity bias: long answers look convincing
            return {"reasoning": f"reference phrase {'found' if correct else 'missing'}; long={long_and_confident}",
                    "score": bool(correct or long_and_confident)}
        return RunnableLambda(judge)

JUDGE_PROMPT = """You are grading a customer-support answer.
Question: {inputs}
Answer: {outputs}
Reference: {reference_outputs}
Does the answer correctly state the reference? Reply with a score of true or false."""

llm_judge = create_llm_as_judge(prompt=JUDGE_PROMPT, judge=StubJudgeModel(), feedback_key="judge_correct")

def judge_correct(inputs: dict, outputs: dict, reference_outputs: dict) -> dict:
    return llm_judge(inputs=inputs["question"], outputs=outputs["answer"], reference_outputs=reference_outputs["answer"])

# ---- 4. A summary evaluator: sees ALL outputs of the experiment at once ----
def overall_mentions_rate(outputs: list[dict], reference_outputs: list[dict]) -> dict:
    hits = [r["answer"].lower() in o["answer"].lower() for o, r in zip(outputs, reference_outputs)]
    return {"key": "overall_mentions_rate", "score": float(np.mean(hits))}

# Try the evaluators on one hand-made case before running the experiment (cheap debugging habit).
sample_out = {"answer": "You can return items within 60 days.", "intent": "returns"}
sample_ref = {"answer": "30 days", "intent": "returns"}
print(mentions_reference(sample_out, sample_ref))
print(intent_correct(sample_out, sample_ref))
print(judge_correct({"question": "q"}, sample_out, sample_ref))

**Reading those three lines.** For the answer *"within 60 days"* against the reference *"30 days"*: the code check says `False` (correct, 60 is wrong), the intent matches (`True`),
and the judge also says `False` (`reference phrase missing`). Always test evaluators on a case you understand before trusting them on 36 runs.

## 4. Run the experiment

One call does everything in the flow diagram from section 0:

```python
evaluate(target, data=examples, evaluators=[...], summary_evaluators=[...], num_repetitions=3, upload_results=False)
```

- `upload_results=False` keeps it local.
- `num_repetitions=3` retakes the exam three times, because agents are random.
- `max_concurrency=1` runs one at a time so our simulated flakiness is reproducible.

In [ ]:
def run_experiment(target, name):
    _calls.clear()                                    # reset the flaky-answer counters so each experiment is reproducible
    return evaluate(
        target,
        data=examples,
        evaluators=[mentions_reference, intent_correct, judge_correct],
        summary_evaluators=[overall_mentions_rate],
        experiment_prefix=name,
        num_repetitions=3,
        upload_results=False,
        max_concurrency=1,
    )

res_v1 = run_experiment(target_v1, "support-bot-v1")
res_v2 = run_experiment(target_v2, "support-bot-v2")

In [ ]:
def to_frame(res):
    df = res.to_pandas()
    keep = {"inputs.question": "question", "outputs.answer": "answer",
            "feedback.mentions_reference": "mentions", "feedback.exact_match": "intent_ok", "feedback.judge_correct": "judge"}
    df = df[[c for c in keep if c in df.columns]].rename(columns=keep)
    for c in ["mentions", "intent_ok", "judge"]:
        df[c] = df[c].astype(float)
    return df

df1, df2 = to_frame(res_v1), to_frame(res_v2)
print("feedback columns LangSmith created:", [c for c in res_v1.to_pandas().columns if c.startswith("feedback")])
print(len(df1), "rows per version = 12 examples x 3 repetitions")
df2.head(6)

**How to read that table.** Each **row** is one run of one example. The three **score columns** (`mentions`, `intent_ok`, `judge`) are the feedback from the three evaluators: `1.0` = pass, `0.0` = fail.
Rows 0 to 5 are the first six questions. Look at the refund row: `mentions` is `0.0` but the judge column (to its right) disagrees. We will come back to that.

## 5. See the results

Numbers in a table are hard to scan. The same data as a **heatmap**: rows are questions, columns are evaluators, colour is the pass rate across the 3 repetitions. Hunt for the red cells.

In [ ]:
METRICS = ["mentions", "intent_ok", "judge"]
LABELS = ["code check\n(mentions)", "intent\n(exact match)", "LLM judge"]
g1 = df1.groupby("question", sort=False)[METRICS].mean()
g2 = df2.groupby("question", sort=False)[METRICS].mean()
short = SHORT

from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list("rg", [C["fail"], "#F6E7B4", C["pass"]])

fig, axes = plt.subplots(1, 2, figsize=(12, 5.6), sharey=True)
for ax, g, title, col in [(axes[0], g1, "v1", C["v1"]), (axes[1], g2, "v2", C["v2"])]:
    im = ax.imshow(g.values, cmap=cmap, vmin=0, vmax=1, aspect="auto")
    ax.set_xticks(range(3)); ax.set_xticklabels(LABELS, fontsize=9)
    ax.set_yticks(range(12)); ax.set_yticklabels(short, fontsize=9)
    for i in range(12):
        for j in range(3):
            v = g.values[i, j]
            ax.text(j, i, f"{v:.2f}".rstrip("0").rstrip("."), ha="center", va="center", fontsize=9,
                    color="white" if v < 0.4 or v > 0.9 else C["ink"])
    ax.set_title(title, color=col, fontsize=13, fontweight="bold")
    ax.spines[:].set_visible(False)
fig.suptitle("Pass rate per question and evaluator (average of 3 repetitions)", fontsize=13, fontweight="bold", x=0.02, ha="left")
plt.tight_layout(); plt.show()

**What the heatmap tells you.**
- **v1 (left):** red in the `code check` column for return window, refund and warranty (the planted hallucinations). One red `intent` cell for customs fees.
- **v2 (right):** return window and warranty turned green. The refund row is still red for the code check, but look at the **judge column in that row**: it is green. That mismatch is the judge being fooled (section 6).
- The two-factor row has a value between 0 and 1 in v2: that is the **flaky** answer.

### Before and after, per question
A heatmap shows levels; a **dumbbell chart** shows *change*. Each row is a question. The blue dot is v1, the teal dot is v2, the line is the movement.

In [ ]:
q1 = df1.groupby("question", sort=False)["mentions"].mean()
q2 = df2.groupby("question", sort=False)["mentions"].mean()
per_q = pd.DataFrame({"v1": q1, "v2": q2}); per_q["change"] = per_q["v2"] - per_q["v1"]
order = per_q.sort_values("change").index

fig, ax = plt.subplots(figsize=(10, 5.4))
for y, q in enumerate(order):
    a, b = per_q.loc[q, "v1"], per_q.loc[q, "v2"]
    col = C["pass"] if b > a else C["fail"] if b < a else C["grey"]
    ax.plot([a, b], [y, y], color=col, lw=3, alpha=0.8, zorder=1)
    ax.scatter([a], [y], color=C["v1"], s=90, zorder=2); ax.scatter([b], [y], color=C["v2"], s=90, zorder=2, marker="D")
ax.set_yticks(range(12)); ax.set_yticklabels([QSHORT[q] for q in order], fontsize=10)
ax.set_xlim(-0.05, 1.05); ax.set_xlabel("pass rate on the code check")
ax.legend(handles=[Patch(color=C["v1"], label="v1 (circle)"), Patch(color=C["v2"], label="v2 (diamond)"),
                   Patch(color=C["pass"], label="improved"), Patch(color=C["fail"], label="got worse")], loc="center left", bbox_to_anchor=(1.01, 0.5), fontsize=9)
ax.set_title("Which questions improved, and which got worse?", loc="left", fontweight="bold")
ax.grid(axis="x", alpha=0.25); plt.tight_layout(); plt.show()

**Read the chart.** Green lines to the right are fixes (return window, warranty). The **red line to the left is a regression**: the two-factor question, caused by the flaky answer. Most questions did not move (grey dots on top of each other).
Averages (v2 is "better") hide that one question got *worse*. This is why the course says: read per-question results, not just the mean.

## 6. Is v2 *really* better? Draw the uncertainty

Our v2 average looks higher, but there are only 12 questions. Could this be luck? The **bootstrap** answers it with a simple idea, shown step by step:

1. Take the 12 per-question changes (v2 minus v1).
2. **Resample** them *with replacement* 12 times, as if re-running the world, and average. That is one fake experiment.
3. Do that 5,000 times. The spread of those 5,000 averages shows how much luck matters.
4. The middle 95% of them is the **95% confidence interval**.

In [ ]:
diff = per_q["change"].to_numpy()
rng = np.random.default_rng(0)
boots = rng.choice(diff, size=(5000, len(diff)), replace=True).mean(axis=1)    # 5,000 fake experiments
mean, lo, hi = bootstrap_ci(diff)

fig, ax = plt.subplots(figsize=(9.5, 4.2))
ax.hist(boots, bins=40, color=C["v2"], alpha=0.75)
ax.set_ylim(0, ax.get_ylim()[1] * 1.25)
ax.axvline(0, color=C["fail"], lw=2.5); ax.text(-0.01, ax.get_ylim()[1] * 0.93, "zero = no change ", color=C["fail"], fontsize=10, fontweight="bold", ha="right")
ax.axvline(lo, color=C["ink"], ls="--"); ax.axvline(hi, color=C["ink"], ls="--")
ax.axvspan(lo, hi, color=C["warn"], alpha=0.12)
ax.text((lo + hi) / 2, ax.get_ylim()[1] * 0.86, f"95% of fake experiments fall between {lo:+.2f} and {hi:+.2f}", ha="center", fontsize=10, bbox=dict(fc="white", ec="none", alpha=0.85))
ax.set_xlabel("average change in pass rate (v2 minus v1)"); ax.set_ylabel("number of fake experiments")
ax.set_title(f"Bootstrap: observed change {mean:+.3f}, but zero is inside the range", loc="left", fontweight="bold")
plt.tight_layout(); plt.show()
print("verdict:", "improvement shown" if lo > 0 else "cannot tell" if hi > 0 else "worse")

**Read the picture.** The red line is *zero change*. Because it sits **inside** the shaded 95% range, a result of "no real improvement" is entirely plausible. The honest verdict is **cannot tell**, even though v2's average is higher.
More questions would make the histogram narrower and, if v2 is truly better, push the whole shaded band to the right of the red line.

## 7. Repetitions: seeing flakiness

Because we ran every question 3 times, we can ask which answers are **reliable**. In the picture each question has three dots, one per run. **Green = pass, red = fail.**
`pass^3` means *all three dots are green*.

In [ ]:
def runs_by_question(df):
    return {q: g["mentions"].tolist() for q, g in df.groupby("question", sort=False)}

r1, r2 = runs_by_question(df1), runs_by_question(df2)
fig, axes = plt.subplots(1, 2, figsize=(12, 5.4), sharey=True)
for ax, runs, title, col in [(axes[0], r1, "v1", C["v1"]), (axes[1], r2, "v2", C["v2"])]:
    for y, d in enumerate(DATASET):
        vals = runs[d["question"]]
        ax.add_patch(Rectangle((0.4, 11 - y - 0.4), 3.2, 0.8, fc="#D4EEDF" if all(v == 1 for v in vals) else "#FBE5BE" if any(v == 1 for v in vals) else "#F9D6DB", ec="none"))
        for k, v in enumerate(vals):
            ax.scatter(k + 1, 11 - y, s=170, color=C["pass"] if v == 1 else C["fail"], zorder=3)
    ax.set_xlim(0.3, 3.7); ax.set_xticks([1, 2, 3]); ax.set_xticklabels(["run 1", "run 2", "run 3"])
    ax.set_yticks(range(12)); ax.set_yticklabels(SHORT[::-1], fontsize=10); ax.tick_params(length=0)
    ax.set_title(title, color=col, fontsize=13, fontweight="bold"); ax.spines[:].set_visible(False)
fig.suptitle("Three runs per question (green = pass, red = fail)", fontsize=13, fontweight="bold", x=0.02, ha="left")
plt.tight_layout(); plt.show()

In [ ]:
def reliability(df):
    g = df.groupby("question", sort=False)["mentions"]
    return float(g.mean().mean()), float((g.min() == 1.0).mean())

for name, d in [("v1", df1), ("v2", df2)]:
    avg, all_pass = reliability(d)
    print(f"{name}: average pass rate {avg:.3f}   |   pass^3 (all 3 runs pass) {all_pass:.3f}")

**Read the dots.** In v2 the two-factor row shows **red, red, green**: it passed only once in three tries. A single run would have shown either all-red or all-green and told you nothing about reliability.
For v2, `pass^3` (0.833) sits below its average pass rate (0.861) *because of that one flaky row*. v1 has no flaky question, so its two numbers match.
The trade-off: for a support agent that users rely on, `pass^3` is the honest number.

## 8. Catching a biased LLM judge, visually

Our stub judge passes any answer longer than 20 words (a *verbosity bias*). We have a code check that acts as **ground truth** for this tiny task, so we can compare them directly.
A **confusion matrix** counts how often the judge and the code check agree.

In [ ]:
both = pd.concat([df1.assign(version="v1"), df2.assign(version="v2")], ignore_index=True)
truth = both["mentions"].astype(bool).to_numpy()
judge = both["judge"].astype(bool).to_numpy()
tpr, tnr = tpr_tnr(judge, truth)
cm = np.array([[(~judge & ~truth).sum(), (judge & ~truth).sum()],
               [(~judge &  truth).sum(), (judge &  truth).sum()]])

fig, axes = plt.subplots(1, 2, figsize=(12, 4.4), gridspec_kw={"width_ratios": [1, 1.25]})
ax = axes[0]
ax.imshow(cm, cmap="Blues")
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=20, color="white" if cm[i, j] > cm.max() / 2 else C["ink"])
ax.add_patch(Rectangle((0.5, -0.5), 1, 1, fill=False, ec=C["fail"], lw=3.5))     # highlight: judge PASS but truth BAD
ax.set_xticks([0, 1]); ax.set_xticklabels(["judge: FAIL", "judge: PASS"]); ax.set_yticks([0, 1]); ax.set_yticklabels(["code check: BAD", "code check: GOOD"])
ax.set_title("Judge vs ground truth", fontweight="bold"); ax.spines[:].set_visible(False)
ax.text(1, 0.3, "wrongly passed", ha="center", color=C["fail"], fontsize=9.5, fontweight="bold")

ax = axes[1]; ax.axis("off"); ax.set_xlim(0, 10); ax.set_ylim(0, 6)
wrong = both[(both["judge"] == 1.0) & (both["mentions"] == 0.0)]["answer"].iloc[0]
ax.text(0, 5.6, "The answer the judge wrongly passed:", fontsize=11, fontweight="bold", va="top")
ax.text(0, 4.9, '"' + wrong[:150] + '..."', fontsize=9.5, family="monospace", va="top", wrap=True,
        bbox=dict(fc="#FBE3E6", ec=C["fail"], boxstyle="round,pad=0.5"))
ax.text(0, 2.3, f"TPR {tpr:.2f}: passes good answers.   TNR {tnr:.2f}: fails bad ones only {tnr:.0%} of the time.\nCohen's kappa vs code check: {cohen_kappa(judge, truth):.2f}", fontsize=10.5, va="top")
ax.text(0, 1.1, "It never states how refunds are paid.\nIt is long, so the biased judge liked it.", fontsize=10.5, va="top", color=C["fail"])
plt.tight_layout(); plt.show()

**Read the matrix.** The **red-outlined cell** is the problem: answers that the code check says are *bad* but the judge **passed**. All of them are the same long non-answer from v2, repeated across the 3 runs.
Overall agreement looks high (kappa is good), yet the judge has a blind spot. Overall accuracy hides blind spots, so always **read the cases the judge gets wrong**.

In [ ]:
# The scorecard: how each version looks through each lens. Notice the judge flatters v2 more than the code check does.
card = pd.DataFrame({
    "v1": [df1["mentions"].mean(), df1["intent_ok"].mean(), df1["judge"].mean(), reliability(df1)[1]],
    "v2": [df2["mentions"].mean(), df2["intent_ok"].mean(), df2["judge"].mean(), reliability(df2)[1]],
}, index=["code check", "intent match", "LLM judge", "pass^3 (reliable)"])

ax = card.plot.bar(figsize=(9, 4.2), rot=0, color=[C["v1"], C["v2"]], ylim=(0, 1.1), width=0.7)
for cont in ax.containers:
    ax.bar_label(cont, fmt="%.2f", fontsize=9, padding=2)
ax.set_title("Scorecard: v1 vs v2 through four lenses", loc="left", fontweight="bold"); ax.set_ylabel("score")
ax.annotate("judge flatters v2\n(fooled by the long answer)", xy=(2.18, 0.95), xytext=(2.6, 1.04), fontsize=9, color=C["fail"],
            arrowprops=dict(arrowstyle="-|>", color=C["fail"]))
plt.tight_layout(); plt.show()

## 9. What changes when you connect to LangSmith

Everything above ran on your laptop. The picture of the difference:

In [ ]:
fig, ax = canvas(13, 4.0, (0, 14), (0, 4.2))
box(ax, 0.3, 0.5, 5.6, 3.0, "", C["soft"])
ax.text(3.1, 3.15, "LOCAL (this notebook)", ha="center", fontsize=11, fontweight="bold")
for i, t in enumerate(["dataset in Python objects", "evaluate(..., upload_results=False)", "results in a DataFrame", "nothing leaves your laptop"]):
    ax.text(0.7, 2.6 - i * 0.62, "• " + t, fontsize=10)
box(ax, 8.0, 0.5, 5.7, 3.0, "", "#DCE8F5")
ax.text(10.85, 3.15, "CONNECTED (LangSmith)", ha="center", fontsize=11, fontweight="bold")
for i, t in enumerate(["named, versioned cloud datasets", "traces of every step of every run", "side-by-side experiment comparison UI", "annotation queues, online evaluators"]):
    ax.text(8.4, 2.6 - i * 0.62, "• " + t, fontsize=10)
arrow(ax, 5.9, 2.0, 8.0, 2.0, label="API key +\nyour data leaves", color=C["warn"], dy=0.25)
ax.set_title("Local versus connected", loc="left", fontsize=12, fontweight="bold")
plt.show()

**Privacy first.** Traces contain your prompts, inputs and outputs. Connecting means sending that data to a third party. Check what your traces hold (use the client's anonymizer or `hide_inputs` / `hide_outputs` options) before pointing real user data at it.
Also remember the trap from the top of this notebook: without the sample-rate setting, this SDK tried to upload even with `upload_results=False`.

### Using a real LLM judge
The only change needed in section 3 is the `judge`:

```python
from openevals.llm import create_llm_as_judge
from openevals.prompts import CORRECTNESS_PROMPT          # ready-made prompts also exist for hallucination, conciseness, RAG groundedness, ...

judge = create_llm_as_judge(
    prompt=CORRECTNESS_PROMPT,
    model="anthropic:<model id>",       # a "provider:model" string; needs the provider's API key and package
    feedback_key="correctness",
)
```
(Not run here. Check the openevals README for the current prompts and the exact model string format.)

## 10. Self-check: can you explain these from the pictures alone?
1. In the flow diagram (section 0), which arrow carries the answer key, and why must the *target* not see it?
2. In the heatmap, which cells show the planted bugs? Which cell shows the judge being fooled?
3. In the dumbbell chart, why is a red line to the left more important than an average that went up?
4. In the bootstrap histogram, what does it mean that zero lies inside the shaded range?
5. In the repetition dots, why is "red, red, green" worse than a pass rate of 0.33 suggests?
6. In the confusion matrix, which cell is the dangerous one, and what would you do about it?

## 11. Exercises
1. **Add an evaluator:** write `concise` that fails answers longer than 25 words. Add it to the heatmap. Which version does it penalize?
2. **A harsher judge:** change `StubJudgeModel` so it ignores length. Redraw the confusion matrix: where does the red outline go?
3. **Per-intent chart:** group results by the expected intent and draw a bar chart. Which intent is weakest?
4. **More repetitions:** set `num_repetitions=10` and redraw the dot plot. How many dots does the flaky question show red?
5. **Regression gate:** reuse `regression_gate` from the main notebook on `per_q` to decide whether v2 may ship.
6. **Swap the target:** replace the stub model with a real chat model and keep every evaluator unchanged.
7. **Trajectory evals:** read about the `agentevals` library and sketch an evaluator for the tool-calling agent from the Harness course.
8. **Legacy API:** older LangChain code used `langchain.evaluation` (`load_evaluator("labeled_criteria")`). Find an example and map it to the evaluator functions here.

## 12. Reading list
- LangSmith: evaluate locally without uploading results: https://docs.langchain.com/langsmith/local
- LangSmith evaluation how-to guides: https://docs.smith.langchain.com/evaluation/how_to_guides/local
- openevals (prebuilt evaluators): search "langchain-ai/openevals" on GitHub *(repository name from memory, not re-checked)*
- Back to the main notebook: `evals_learning.ipynb`